## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Shelf positions and spread: Bartlett's and Levene's tests

**Test for Homogeneity of Variance** · a worked example from the StatNex teaching guide

**The story.** A convenience-store chain tested where to put a new energy drink: at eye level, on the top shelf, on the bottom shelf, or by the checkout. Each position was tried for five days, one position after another, and daily cans sold were recorded.

**The question.** Does shelf position change sales, and which positions differ from each other?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/one_way_anova__tukey.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Day,Eye level,Top shelf,Bottom shelf,Checkout
Day 1,49.0,39.0,37.0,47.0
Day 2,50.0,41.0,34.0,50.0
Day 3,57.0,38.0,37.0,49.0
Day 4,51.0,39.0,37.0,47.0
Day 5,54.0,40.0,30.0,47.0
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
groups = ["Eye level", "Top shelf", "Bottom shelf", "Checkout"]

## Descriptive statistics

In [ ]:
# One column per group. Text and blank cells are skipped, so the groups may differ
# in size.
values = [pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float) for name in groups]
descriptive = pd.DataFrame({"n": [len(v) for v in values], "mean": [v.mean() for v in values],
                            "variance": [v.var(ddof=1) for v in values]}, index=groups)
descriptive["SD"] = np.sqrt(descriptive["variance"])
descriptive["SE"] = descriptive["SD"] / np.sqrt(descriptive["n"])
print(descriptive)

## Bartlett's test and Levene's test

In [ ]:
# Tests of equal variances for any number of groups. Bartlett's test compares the log
# of the pooled variance with the groups' own; like the F test it assumes normal data.
# Levene's test is a one-way ANOVA of each value's distance from the centre of its
# group: from the mean (Levene), or from the median (Brown-Forsythe), which a few
# extreme values move less. Neither squares the distances, so heavy tails mislead
# them far less than they mislead Bartlett's test: read them when the shape of the
# data is uncertain.
def levene_test(center):
    # F divides by how much the distances vary inside the groups. Where they do not
    # vary at all -- two values in every group, say -- there is no test, and SciPy
    # would print an infinite F: NaN says so instead. "Not at all" allows for rounding:
    # a sum of squares below (64 x machine epsilon)^2 of the values' own is noise.
    middle = np.median if center == "median" else np.mean
    distances = [np.abs(v - middle(v)) for v in values]
    within = sum(((d - d.mean()) ** 2).sum() for d in distances)
    if within <= (64 * np.finfo(float).eps) ** 2 * sum((v**2).sum() for v in values):
        return np.nan, np.nan
    return tuple(stats.levene(*values, center=center))


k, total = len(values), sum(len(v) for v in values)
bartlett = stats.bartlett(*values)
levene = levene_test("mean")
brown_forsythe = levene_test("median")
variance_tests = pd.DataFrame(
    [[bartlett.statistic, k - 1, np.nan, bartlett.pvalue],
     [levene[0], k - 1, total - k, levene[1]],
     [brown_forsythe[0], k - 1, total - k, brown_forsythe[1]]],
    index=["Bartlett", "Levene (mean)", "Levene (median; Brown-Forsythe)"],
    columns=["statistic", "df1", "df2", "P"])
print()
print(variance_tests)

## What the result says

One-way ANOVA assumes that daily sales vary about as much at every shelf position. This run checks that assumption. The variance, a measure of spread, is 10.7 at eye level and 9.5 on the bottom shelf, but only 1.3 on the top shelf and 2 at checkout.

Bartlett's test compares the four variances directly. Its chi-square statistic is 5.45 and its P value is 0.141. The P value is the chance of variances this unequal if all four positions really varied the same. Levene's test uses each day's distance from the mean of its position, and gives P = 0.054. Its Brown-Forsythe form uses the distance from the median, the middle value, and gives P = 0.576.

None of the three tests is significant, so these data do not show that the spreads differ. That is not proof that they are equal: 5 days per position can easily miss a real difference. So it is safer to read Welch's test as well, in the one-way ANOVA.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.